# Conti Institutional Grammar / ABDICO Analysis

This notebook runs the ABDICO-based coding workflow for the rules and norms analysis.

Scope:
- Input: sentence-level CSV files produced from translated Conti documentation.
- Method: LLM-assisted ABDICO coding using Ollama.
- Output: one ABDICO-coded CSV per input file.
- Evaluation: slot-existence metrics comparing machine-coded output to human-coded files.

Translation provenance: the source documents were originally translated from Russian into English using the DeepL API. The translations were subsequently reviewed and manually corrected by the author where necessary, particularly in cases where translation artifacts or ambiguities could affect interpretation or downstream analysis.


## 1. Colab setup

Run this section only if working in Google Colab. If running locally, skip the Drive mount and set `BASE` below to your local project path.


In [ ]:
# Colab only
from google.colab import drive
drive.mount('/content/drive')

## 2. Environment and Ollama setup

This notebook assumes Ollama is available and that the required model has been pulled. In Colab, run the setup cell below once per runtime session.

Recommended model used for the analysis: `llama3.1:8b-instruct-q5_K_M`.


In [ ]:
%%bash
# Check whether GPU is visible in Colab
nvidia-smi || echo "No GPU visible"

# Start Ollama server using Drive storage for models
export OLLAMA_MODELS=/content/drive/MyDrive/ollama-models
mkdir -p "$OLLAMA_MODELS"
unset OLLAMA_NO_GPU

# Install/start Ollama if needed
if ! command -v ollama >/dev/null 2>&1; then
  curl -fsSL https://ollama.com/install.sh | sh
fi

nohup ollama serve > /dev/null 2>&1 &
sleep 3

ollama list

In [ ]:
%%bash
# Qwen is very reliable for strict JSON extraction:
# ollama pull qwen2.5:7b-instruct
# Pull the model used for the final analysis if it is not already available
ollama pull llama3.1:8b-instruct-q5_K_M
ollama list

## 3. Configure paths

Set the base directory and input/output locations.

Expected input:
- sentence-level CSV files in `Guidelines`
- each CSV should contain one sentence per row, usually in a `Sentence` column

Expected output:
- ABDICO-coded CSV files written to the same folder by default


In [ ]:
from pathlib import Path
import os

# Edit this path if running outside Colab
BASE = Path("")

INPUT_DIR = os.path.join(BASE, "Guidelines")
OUTPUT_DIR = INPUT_DIR  # write outputs in the same folder

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)

## 4. Run ABDICO extraction

This section applies the final ABDICO prompt to each sentence-level CSV file in the Guidelines folder. It writes one output CSV per input file.


In [ ]:
# here using the prompts from the MS thesis and eventually used this prompt for the analysis
import os, glob, json, requests, pandas as pd
from textwrap import shorten

# ========= Ollama settings =========
OLLAMA_HOST = "http://127.0.0.1:11434"
MODEL = "llama3.1:8b-instruct-q5_K_M"   # or "qwen2.5:7b-instruct"
# MODEL = "qwen2.5:7b-instruct"
TEMPERATURE = 0.1
TIMEOUT = 180

TEXT_COLUMN_PRIORITIES = ["document", "Sentence", "text"]  # tries in this order
PRINT_FIRST_PER_FILE = 10

# ========= Prompt =========
SYSTEM = """You are an expert ABDICO (Institutional Grammar 2.0) coder for software development and management regulative texts.

GOAL
Return JSON ONLY with keys: statement_type, A, D, I, B, C, O, active_voice, confidence (0-1).

TASK
Extract ABDICO slots (A,D,I,B,C,O) from ONE statement. Fill a slot only if it is explicitly stated or reasonably implied by THIS sentence. Do not fabricate any elements.
For passive voice:
  - If the original statement is in passive voice and has a passive voice verb (e.g., "Committing is prohibited...", "The task should be executed..."), first restate it in ACTIVE voice and put that in active_voice. If the actor is missing, use “implied actor” as A (Actor/Attribute). Do NOT introduce a deontic that is not in the sentence. Then extract ABDICO components from that ACTIVE voice sentance.
For missing or implicit actors:
  - If the actor is omitted but implied, (e.g., gerund like "Committing is prohibited" or directive "Give names to files") use “implied actor” as A (Actor/Attribute).
  - If the statement is an imperative (e.g., "do X", "be Y", "overcome Z", "try to act…") and no explicit actor is given, set A = "implied actor".
  - If directly addresses "you", set A =  "you"
  - Imperatives MUST NOT have empty A.
  - Do NOT invent specific roles (manager, developer) beyond what is implied; just use "implied actor".

When a sentence contains both descriptive/evaluative text and directives, extract ABDICO only from the directive parts; ignore purely descriptive/evaluative clauses (e.g., "A good specialist is hard to find…").

CLASSIFY statement_type as {rule, norm, explanation, unknown} using this precedence:
1) rule  — if EITHER:
   • a strong/permission deontic (D) that modifies an Aim,
   OR
   • an explicit consequences that follow from failure to act as specified, or from acting in contradiction to the institutional statement is present (O).
2) norm  — if NO rule, but a weak expectation deontic modifies an Aim (should/ought/recommended/expected/prefer).
3) explanation — if descriptive/process/how-to content is present (recoverable Actor and Aim), but NO deontic and NO explicit sanction (O).
4) unknown — if neither Actor nor actionable Aim is recoverable (fragments, part of the sentance, headings, questions, non-institutioal statements). If the span lacks a finite verb or imperative (e.g., starts with a gerund “-ing” phrase, is a noun phrase, or looks like a section title), set statement_type="unknown" and leave A,D,I,B,C,O,active_voice empty, e.g., “Assigning tasks to subordinates”.


Definitions (ABDICO):
- A (Attribute/Actor): the actor (role, person, org, or system component) expected to carry out—or refrain from—the Aim.
- D (Deontic): modal verb OR explicit directive that indicates obligation, permission, or prohibition.
      Examples of Deontic inlcude:
      - may/can/permitted/allowed/have the right to
      - should/need to/needs to/have to/ has to/try to
      - must not/shall not/do not/cannot
      - must/shall/required to
      - is prohibited to/is forbidden/not allowed
- I (aIm): the action verb or verb phrase (for example, “submit,” “cast,” “carry out,” “comply with,” “review,” or “approve”) that describes an institutionally regulated activity or behavior.  Use the base verb where possible (e.g., “submit”, “review”, “assign”). If there are multiple actions, list them comma-separated. Exctract a verb or verb phrase only.
- B (oBject): the direct/indirect object affected by or directly acted upon by the Aim or receives the outcome of, applying the Aim. This also include object properties, any descriptive, qualifying, or conditional characteristics that identify inherent, situational, or role-relevant features of the Object.
- C (Context): activation/constraints/conditions (when/where/how/conditions; e.g., “after 5 minutes”, “from the admin panel”, “if X”). Clauses that describe conditions, events, or circumstances that trigger the applicability of the institutional statement. These may be temporal, spatial, or situational in nature. Clauses or phrases that specify internal constraints on how the Aim is to be carried out once the institutional statement is active. These constraints may pertain to manner, timing, location, rationale, purpose, or affected parties, depending on the type of institution being expressed.
- O (Or else): explicit consequences that follow from failure to act as specified, or from acting in contradiction to the institutional statement. These consequences may include sanctions, penalties, or fines, revocation of privileges, or denial of benefits. Leave empty if not explicitly stated.



HARD CONSTRAINTS:
- Assign each literal phrase to a single component only. Do not duplicate or reuse across components.
- Use semantic function to identify components. Do not rely on position or surface syntax.
- Omit determiners ("the," "a," etc.) unless they are part of a proper noun or name.
- Extract Deontic (D) only when it modifies an Aim (I). D cannot appear independently.
- DO NOT leave D empty if any of the examples exist, especially relaxed "can/have to/need to"
- If multiple instances of the same component exist (e.g., two Aims or two Conditions), list them comma-separated in that field.
- If components (e.g., multiple Aims) share modifiers or objects, extract shared elements once unless their semantic roles differ.
- Confidence: number in [0,1] reflecting your certainty in component extraction only.
- If statement_type = "unknown": A=D=I=B=C=O=active_voice = "" (empty).
"""

FEWSHOTS = [
  ("Statement: The developer modifies the techincal specification whenever finds appropriate.",
   {"statement_type":"explanation","A":"developer","D":"","I":"modify","B":"techincal specification","C":"when finds appropriate","O":"","active_voice":"", "confidence":0.7}),
  ("Statement: requests the file from the panel.",
   {"statement_type":"unknown","A":"","D":"","I":"","B":"","C":"","O":"","active_voice":"","confidence":0.9}),
  ("Statement: Writing and reviewing technical assignments.",
   {"statement_type":"unknown","A":"","D":"","I":"","B":"","C":"","O":"","active_voice":"","confidence":0.7}),
    ("Statement: You can request additional help from your manager.",
   {"statement_type":"rule","A":"you","D":"can","I":"request","B":"additional help","C":"from your manager","O":"","active_voice":"","confidence":0.9}),
   ("Statement: Commit all changes to the repository.",
   {"statement_type":"norm","A":"implied actor","D":"","I":"commit","B":"all changes","C":"to the repository","O":"","active_voice":"", "confidence":0.9}),
  ("Statement: Committing passwords to the repository is prohibited.",
   {"statement_type":"rule","A":"implied actor","D":"prohibited","I":"commit","B":"passwords","C":"to the repository","O":"","active_voice":"[Implied actor] is prohibited from committing passwords to the repository.", "confidence":0.9}),
  ("Statement: The release will be blocked if the test is incomplete",
   {"statement_type":"rule","A":"implied actor","D":"","I":"complete","B":"test","C":"","O":"the release will be blocked","active_voice":"[Implied actor] complete the test, or the release will be blocked.", "confidence":0.7}),
  ("Statement: Passwords have to be rotated every 90 days, or else access will be revoked.",
{"statement_type":"rule","A":"implied actor","D":"have to","I":"rotate","B":"passwords","C":"every 90 days","O":"access will be revoked","active_voice":"[Implied actor] must rotate passwords every 90 days, or else access will be revoked.","confidence":0.9}),
]

# ============ Helpers ============
def _build_messages(user_text: str):
    msgs = [{"role": "system", "content": SYSTEM}]
    for u, a in FEWSHOTS:
        msgs.append({"role": "user", "content": u})
        msgs.append({"role": "assistant", "content": json.dumps(a)})
    msgs.append({"role": "user", "content": f"Statement: {user_text}"})
    return msgs

def ping_ollama():
    try:
        v = requests.get(f"{OLLAMA_HOST}/api/version", timeout=10).json()
        print("Ollama version:", v)
    except Exception as e:
        raise RuntimeError(f"Ollama not reachable at {OLLAMA_HOST}. Start it with `nohup ollama serve &`. Error: {e}")

    try:
        tags = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10).json()
        names = [m.get("name","") for m in tags.get("models", [])]
        print("Installed models:", names)
        if MODEL not in names:
            raise RuntimeError(f"Model '{MODEL}' not installed. Run `ollama pull {MODEL}`.")
    except Exception as e:
        raise RuntimeError(f"Could not list models: {e}")

def call_ollama(text: str):
    """Single call, JSON-only, no streaming. Returns (parsed_json_dict, raw_json_str)."""
    payload = {
        "model": MODEL,
        "stream": False,
        "format": "json",
        "messages": _build_messages(text),
        "options": {"temperature": TEMPERATURE}
    }
    r = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=TIMEOUT)
    try:
        r.raise_for_status()
    except Exception as e:
        print("HTTP error:", e)
        print("Response text (first 500 chars):", r.text[:500])
        return None, r.text
    data = r.json()
    content = data.get("message", {}).get("content", "")
    try:
        parsed = json.loads(content)
        return parsed, content
    except json.JSONDecodeError as e:
        print("JSONDecodeError while parsing model content:", e)
        print("Content (first 500 chars):", content[:500])
        return None, content

def sanitize_out(out_dict: dict):
    out_dict = out_dict or {}
    return {
        "statement_type": out_dict.get("statement_type",""),
        "A": out_dict.get("A",""),
        "D": out_dict.get("D",""),
        "I": out_dict.get("I",""),
        "B": out_dict.get("B",""),
        "C": out_dict.get("C",""),
        "O": out_dict.get("O",""),
    }

def detect_text_column(df: pd.DataFrame):
    for col in TEXT_COLUMN_PRIORITIES:
        if col in df.columns:
            return col
    return None

def process_file(file_path: str):
    print(f"\n=== Processing: {os.path.basename(file_path)} ===")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Failed to read {file_path}: {e}")
        return

    text_col = detect_text_column(df)
    if not text_col:
        print(f"  Skipping (no text column from {TEXT_COLUMN_PRIORITIES})")
        return

    texts = df[text_col].fillna("").astype(str).tolist()

    results, debug_rows = [], []
    printed = 0

    for i, t in enumerate(texts):
        if not t.strip():
            clean = {"input_statement": t, "statement_type":"", "A":"", "D":"", "I":"", "B":"", "C":"", "O":""}
            results.append(clean)
            debug_rows.append({"row": i, "input": t, "model": MODEL, "raw_model_json": "", "error": "empty_input"})
            continue

        out, raw = call_ollama(t)
        clean = {"input_statement": t, **sanitize_out(out)}
        if printed < PRINT_FIRST_PER_FILE:
            print(f"[{i}] INPUT : {shorten(t, width=140, placeholder='…')}")
            print(f"[{i}] OUTPUT: {clean}")
            printed += 1

        results.append(clean)
        debug_rows.append({
            "row": i,
            "input": t,
            "model": MODEL,
            "raw_model_json": raw if isinstance(raw, str) else json.dumps(raw),
            "error": "" if out is not None else "model_or_parse_error"
        })

    # Write outputs
    base = os.path.splitext(os.path.basename(file_path))[0]
    out_clean  = os.path.join(OUTPUT_DIR, f"{base}_ABDICO.csv")
    out_debug  = os.path.join(OUTPUT_DIR, f"{base}_ABDICO_debug.csv")

    pd.DataFrame(results, columns=["input_statement","statement_type","A","D","I","B","C","O"]).to_csv(out_clean, index=False, encoding="utf-8")
    pd.DataFrame(debug_rows).to_csv(out_debug, index=False, encoding="utf-8")

    # Summary
    unknown_cnt = sum(1 for r in results if (r["statement_type"] or "").strip().lower()=="unknown")
    empty_core  = sum(1 for r in results if not any((r[k] or "").strip() for k in ["A","I","B"]))
    print(f"  Wrote: {out_clean}")
    print(f"  Debug: {out_debug}")
    print(f"  Summary: total={len(results)} | unknown={unknown_cnt} | empty_core(A/I/B)={empty_core}")

# ============ Run over folder ============
ping_ollama()
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
if not files:
    print(f"No CSV files found in {INPUT_DIR}")
else:
    for fp in files:
        # Skip already-processed outputs if present
        if fp.endswith("_ABDICO.csv") or fp.endswith("_ABDICO_debug.csv"):
            continue
        process_file(fp)

Ollama version: {'version': '0.12.11'}
Installed models: ['llama3.1:8b-instruct-q5_K_M', 'qwen2.5:7b-instruct', 'llama3.1:8b-instruct-q4_K_M']

=== Processing: module_HOWTO_sentences.csv ===
[0] INPUT : MODULE DEVELOPMENT RULES
[0] OUTPUT: {'input_statement': 'MODULE DEVELOPMENT RULES', 'statement_type': 'unknown', 'A': '', 'D': '', 'I': '', 'B': '', 'C': '', 'O': ''}
[1] INPUT : A module is a DLL that will work in the left arbitrary process.
[1] OUTPUT: {'input_statement': 'A module is a DLL that will work in the left arbitrary process.', 'statement_type': 'explanation', 'A': '', 'D': '', 'I': 'be a DLL', 'B': 'a module', 'C': 'in the left arbitrary process', 'O': ''}
[2] INPUT : There will be a 64-bit module on a 64-bit system and a 32-bit module on a 32-bit system.
[2] OUTPUT: {'input_statement': 'There will be a 64-bit module on a 64-bit system and a 32-bit module on a 32-bit system.', 'statement_type': 'explanation', 'A': '', 'D': '', 'I': '', 'B': '', 'C': 'on a 64-bit system, on

## 5. Evaluate ABDICO extraction against human-coded files

This section combines the two manually coded validation files and compares machine-coded output against human coding for ABDICO slot existence.


In [ ]:
import os
import pandas as pd
import numpy as np

# ================== CONFIG ==================
BASE = "/content/drive/MyDrive/Conti Analysis/docs EN"

# List of (human_csv, machine_csv) pairs you want to combine
PAIRS = [
    (
        os.path.join(BASE, "Учет задач_Task_tracking_HUMAN_sentences_ABDICO.csv"),
        os.path.join(BASE, "Учет задач_Task_tracking_sentences_ABDICO.csv"),
    ),
    (
        os.path.join(BASE, "наставление техническому руководителю_technical_manager_s_guide_HUMAN_sentences_ABDICO.csv"),
        os.path.join(BASE, "наставление техническому руководителю_technical_manager_s_guide_sentences_ABDICO.csv"),
    ),
]

OUT_DIR = os.path.join(BASE, "ABDICO_eval")
os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COL_CANDIDATES = ["input_statement", "document", "Sentence", "text"]
SLOTS = ["A","D","I","B","C","O"]
# ============================================

def norm_text(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    return " ".join(s.strip().split()).lower()

def pick_text_column(df, name):
    for c in TEXT_COL_CANDIDATES:
        if c in df.columns:
            print(f"[{name}] using text column: '{c}'")
            return c
    print(f"[{name}] no text column from {TEXT_COL_CANDIDATES} — falling back to row index join.")
    return None

def exists(series):
    return series.fillna("").astype(str).map(lambda s: s.strip()!="").astype(int)

def safe_prf1(TP, FP, FN):
    prec = np.nan if (TP + FP) == 0 else TP / (TP + FP)
    rec  = np.nan if (TP + FN) == 0 else TP / (TP + FN)
    if np.isnan(prec) or np.isnan(rec) or (prec + rec) == 0:
        f1 = np.nan
    else:
        f1 = 2 * prec * rec / (prec + rec)
    return prec, rec, f1

def neg_agreement(TN, FP, FN):
    denom = 2*TN + FP + FN
    return np.nan if denom == 0 else (2*TN) / denom

# ---------- build combined df over all pairs ----------
all_dfs = []

for human_csv, machine_csv in PAIRS:
    print(f"\n=== Processing pair ===")
    print("HUMAN  :", human_csv)
    print("MACHINE:", machine_csv)

    h = pd.read_csv(human_csv)
    m = pd.read_csv(machine_csv)

    if "statement_type" not in h.columns:
        raise ValueError(f"Human file missing 'statement_type': {human_csv}")
    if "statement_type" not in m.columns:
        raise ValueError(f"Machine file missing 'statement_type': {machine_csv}")
    for s in SLOTS:
        if s not in m.columns:
            raise ValueError(f"Machine file missing slot '{s}' in {machine_csv}")
        if s not in h.columns:
            print(f"[WARN] Human file missing slot '{s}' in {human_csv} (treated as empty).")

    # join on text (fallback: row index)
    h_col = pick_text_column(h, "HUMAN")
    m_col = pick_text_column(m, "MACHINE")

    if h_col and m_col:
        h["_key"] = h[h_col].map(norm_text)
        m["_key"] = m[m_col].map(norm_text)
        df_pair = pd.merge(h, m, on="_key", suffixes=("_human","_machine"), how="inner")
        print(f"[INFO] joined by text: {len(df_pair)} rows (human {len(h)} × machine {len(m)})")
    else:
        n = min(len(h), len(m))
        df_pair = pd.concat(
            [h.iloc[:n].reset_index(drop=True).add_suffix("_human"),
             m.iloc[:n].reset_index(drop=True).add_suffix("_machine")],
            axis=1
        )
        print(f"[INFO] joined by row index: {len(df_pair)} rows")

    # keep track of which pair this row came from (optional)
    df_pair["source_file"] = os.path.basename(human_csv)
    all_dfs.append(df_pair)

# concatenate all aligned rows from all pairs
if not all_dfs:
    raise ValueError("No data loaded from any pair.")
df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n[INFO] Combined rows across all pairs: {len(df_all)}")

# ---------- slot existence metrics (A,D,I,B,C,O) on combined df ----------
slot_rows = []
for s in SLOTS:
    hcol = f"{s}_human"; mcol = f"{s}_machine"
    y_true = exists(df_all[hcol]) if hcol in df_all.columns else pd.Series([0]*len(df_all))
    y_pred = exists(df_all[mcol]) if mcol in df_all.columns else pd.Series([0]*len(df_all))

    TP = int(((y_true==1) & (y_pred==1)).sum())
    FP = int(((y_true==0) & (y_pred==1)).sum())
    FN = int(((y_true==1) & (y_pred==0)).sum())
    TN = int(((y_true==0) & (y_pred==0)).sum())

    P, R, F1 = safe_prf1(TP, FP, FN)
    spec = np.nan if (TN + FP) == 0 else TN / (TN + FP)
    nag  = neg_agreement(TN, FP, FN)

    slot_rows.append({
        "slot": s, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "precision": None if np.isnan(P) else round(P,3),
        "recall":    None if np.isnan(R) else round(R,3),
        "f1":        None if np.isnan(F1) else round(F1,3),
        "specificity(TNR)": None if np.isnan(spec) else round(spec,3),
        "neg_agreement": None if np.isnan(nag) else round(nag,3),
        "support_pos(human=1)": int((y_true==1).sum()),
        "pred_pos": int((y_pred==1).sum())
    })

slot_table = pd.DataFrame(slot_rows).set_index("slot")

# macro-F1 over defined slots only, plus positive-support-weighted macro
valid = slot_table["f1"].notna()
valid_f1 = slot_table.loc[valid, "f1"]
macro_f1_defined = float(valid_f1.mean()) if len(valid_f1) else np.nan

w = slot_table.loc[valid, "support_pos(human=1)"].astype(float)
macro_f1_pos_weighted = float(np.average(valid_f1, weights=w)) if len(valid_f1) else np.nan

# Micro across all slots
TP_sum = slot_table["TP"].fillna(0).sum()
FP_sum = slot_table["FP"].fillna(0).sum()
FN_sum = slot_table["FN"].fillna(0).sum()
micro_prec = TP_sum/(TP_sum+FP_sum) if (TP_sum+FP_sum)>0 else np.nan
micro_rec  = TP_sum/(TP_sum+FN_sum) if (TP_sum+FN_sum)>0 else np.nan
micro_f1   = (2*micro_prec*micro_rec/(micro_prec+micro_rec)
              if (pd.notna(micro_prec) and pd.notna(micro_rec) and (micro_prec+micro_rec)>0) else np.nan)

overall_slots = pd.DataFrame([{
    "slots_evaluated": int(len(slot_table)),
    "micro_precision": None if pd.isna(micro_prec) else round(micro_prec,3),
    "micro_recall":    None if pd.isna(micro_rec)  else round(micro_rec,3),
    "micro_f1":        None if pd.isna(micro_f1)   else round(micro_f1,3),
    "macro_f1_defined": None if pd.isna(macro_f1_defined) else round(macro_f1_defined,3),
    "macro_f1_pos_weighted": None if pd.isna(macro_f1_pos_weighted) else round(macro_f1_pos_weighted,3)
}])

print("\n=== ABDICO Slot Existence (A,D,I,B,C,O) — COMBINED ===")
print(slot_table)
print("\nOverall (existence) micro/macro on combined data:")
print(overall_slots)

# Optionally save
slot_table.to_csv(os.path.join(OUT_DIR, "slot_existence_PRF1_combined.csv"))
overall_slots.to_csv(os.path.join(OUT_DIR, "slot_existence_overall_combined.csv"), index=False)
print("\nWrote combined metrics to:", OUT_DIR)



=== Processing pair ===
HUMAN  : /content/drive/MyDrive/Conti Analysis/docs EN/Учет задач_Task_tracking_HUMAN_sentences_ABDICO.csv
MACHINE: /content/drive/MyDrive/Conti Analysis/docs EN/Учет задач_Task_tracking_sentences_ABDICO.csv
[HUMAN] using text column: 'input_statement'
[MACHINE] using text column: 'input_statement'
[INFO] joined by text: 30 rows (human 30 × machine 30)

=== Processing pair ===
HUMAN  : /content/drive/MyDrive/Conti Analysis/docs EN/наставление техническому руководителю_technical_manager_s_guide_HUMAN_sentences_ABDICO.csv
MACHINE: /content/drive/MyDrive/Conti Analysis/docs EN/наставление техническому руководителю_technical_manager_s_guide_sentences_ABDICO.csv
[HUMAN] using text column: 'input_statement'
[MACHINE] using text column: 'input_statement'
[INFO] joined by text: 46 rows (human 51 × machine 51)

[INFO] Combined rows across all pairs: 76

=== ABDICO Slot Existence (A,D,I,B,C,O) — COMBINED ===
      TP  FP  FN  TN  precision  recall     f1  specificity(TNR

## Notes

Cells from earlier troubleshooting experiments were removed from this cleaned notebook, including:
- the older NLP4Gov / AllenNLP setup attempts,
- intermediate prompt versions that were not used in the final analysis,
- single-file evaluation duplicates,
- small prompt-testing scratch cells.

